# Part 5: The ”News Feed” Extraction Pipeline (30 Points)
- The Goal: Convert a raw text feed of mixed items into a structured Excel report.
- Input: data/news_feed.txt (30 mixed items).
- Output: output/flood_report.xlsx
- Instructions:
    - Define Schema: Create a Pydantic model CrisisEvent with:
      - district (Literal: Colombo, Gampaha, Kandy, Kalutara, Galle, …)
      - flood_level_meters (float/None)
      - vicLm_count (int, default 0)
      - main_need (str)
      - status (Literal: Critical, Warning, Stable)
    - Process Feed:
      - Load lines from data/news_feed.txt.
      - For each line:
            - Extract JSON using json_extract.v1.
            - Validate using CrisisEvent.model_validate_json().
            - If valid, add to list. If invalid, skip and log warning.
    - Save: Convert valid objects to a Pandas DataFrame and save as flood_report.xlsx.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

import sys
sys.path.append('..')

from utils.token_utils import pick_encoding, count_text_tokens
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
import tiktoken
import pandas as pd
from IPython.display import Markdown, display
from utils.json_utils import format_pydantic_schema_for_prompt, parse_json_with_pydantic
import json

In [41]:
# pick a model
model = pick_model('groq', 'reason')
print(f'Using model: {model}')

client = LLMClient('groq', model)

Using model: openai/gpt-oss-safeguard-20b


In [68]:
class CrisisEvent(BaseModel):
    district: Literal[
        'Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo',
        'Galle', 'Gampaha', 'Hambantota', 'Jaffna', 'Kalutara',
        'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar',
        'Matale', 'Matara', 'Monaragala', 'Mullaitivu', 'Nuwara Eliya',
        'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya'
    ] = Field(..., description="District affected by the crisis event")
    flood_level_meters: float | None = Field(..., description="Flood level in meters, if applicable")
    victim_count: int = Field(..., ge=0, description="Number of victims affected")
    main_need: str = Field(..., min_length=1, description="Main need of the affected population")
    status: Literal["Critical", "Warning", "Stable"] = Field(..., description="Status of the crisis event")

schema_str = format_pydantic_schema_for_prompt(CrisisEvent)
print(schema_str)

{
  "properties": {
    "district": {
      "description": "District affected by the crisis event",
      "enum": [
        "Ampara",
        "Anuradhapura",
        "Badulla",
        "Batticaloa",
        "Colombo",
        "Galle",
        "Gampaha",
        "Hambantota",
        "Jaffna",
        "Kalutara",
        "Kandy",
        "Kegalle",
        "Kilinochchi",
        "Kurunegala",
        "Mannar",
        "Matale",
        "Matara",
        "Monaragala",
        "Mullaitivu",
        "Nuwara Eliya",
        "Polonnaruwa",
        "Puttalam",
        "Ratnapura",
        "Trincomalee",
        "Vavuniya"
      ],
      "title": "District",
      "type": "string"
    },
    "flood_level_meters": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "description": "Flood level in meters, if applicable",
      "title": "Flood Level Meters"
    },
    "victim_count": {
      "description": "Number of victim

In [61]:
with open('../data/News Feed.txt', 'r') as f:
    news_feed = [line.strip() for line in f.readlines()]
    print(f"Total news items: {len(news_feed)}")
    for i in range(5):
        print(news_feed[i])

Total news items: 30
BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.
SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.
URGENT: Landslide in Kalutara. 12 people missing. Rescue team needed.
Gampaha town center is fully underwater. Flood level est 2.0 meters. 500 people displaced to temple. Need dry rations.


In [79]:
valid_news_feed_list = []

for news in news_feed:
    print(news)
    prompt_text, spec = render("json_extract.v1", schema=schema_str, text=news)
    response = client.json_chat([
        {
            "role": "user",
            "content": prompt_text
        }],
        temperature=0.0
    )
    success, data, err = parse_json_with_pydantic(response['text'], CrisisEvent)
    if success:
        print(f"Valid JSON extracted for news: {news[:50] + '...' if len(news) > 50 else news}\n")
        valid_news_feed_list.append(data)
        log_llm_call('groq', model, 'json_extract.v1', response['latency_ms'], response['usage'])
    else:
        print(f"Invalid JSON extracted for news: {news[:50] + '...' if len(news) > 50 else news}")
        print(f"Skipping this news item. Error: {err}\n")
        log_llm_call('groq', model, 'json_extract.v1', response['latency_ms'], response['usage'], notes=f"Invalid JSON: {err}")

BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.
Valid JSON extracted for news: BREAKING: Water levels in Kelani River (Colombo) h...

SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
Valid JSON extracted for news: SOS: 5 people trapped on a roof in Ja-Ela (Gampaha...

Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.
Valid JSON extracted for news: Update: Kandy road cleared near Peradeniya. Traffi...

URGENT: Landslide in Kalutara. 12 people missing. Rescue team needed.
Valid JSON extracted for news: URGENT: Landslide in Kalutara. 12 people missing. ...

Gampaha town center is fully underwater. Flood level est 2.0 meters. 500 people displaced to temple. Need dry rations.
Valid JSON extracted for news: Gampaha town center is fully underwater. Flood lev...

Just saw a navy boat in Colombo. Good job guys.
Valid JSON extracted for news: Just saw a nav

In [80]:
print(f"Total valid news feed items extracted: {len(valid_news_feed_list)}")
print("\nSample valid news feed items:\n")
for item in valid_news_feed_list[:5]:
    print(item.model_dump_json())

Total valid news feed items extracted: 30

Sample valid news feed items:

{"district":"Colombo","flood_level_meters":9.5,"victim_count":0,"main_need":"flood relief","status":"Critical"}
{"district":"Gampaha","flood_level_meters":null,"victim_count":5,"main_need":"Boat","status":"Critical"}
{"district":"Kandy","flood_level_meters":null,"victim_count":0,"main_need":"Road safety","status":"Stable"}
{"district":"Kalutara","flood_level_meters":null,"victim_count":12,"main_need":"Rescue team needed","status":"Critical"}
{"district":"Gampaha","flood_level_meters":2.0,"victim_count":500,"main_need":"dry rations","status":"Critical"}


In [81]:
data = [item.model_dump() for item in valid_news_feed_list]
df = pd.DataFrame(data)
df['flood_level_meters'] = df['flood_level_meters'].fillna('None')
df

,district,flood_level_meters,victim_count,main_need,status
0,Colombo,9.5,0,flood relief,Critical
1,Gampaha,None,5,Boat,Critical
2,Kandy,None,0,Road safety,Stable
3,Kalutara,None,12,Rescue team needed,Critical
4,Gampaha,2.0,500,dry rations,Critical
5,Colombo,None,0,None,Stable
6,Matara,None,0,monitoring,Stable
7,Colombo,1.5,1,Rescue,Critical
8,Galle,None,0,No immediate need,Stable
9,Gampaha,None,0,mosquito control measures,Warning


In [82]:
df.to_excel('../output/flood_report.xlsx', index=False)
print("DataFrame saved to '../output/flood_report.xlsx'")

DataFrame saved to '../output/flood_report.xlsx'
